# Extract per-video VGGish audio embeddings (First Impressions dataset)

Produces one `<audio_output>/<split>/<video_id>.npy` (15, 128) float32 per valid video.

Pipeline (reference: `vid_to_wav.ipynb` + `wav_to_vec.ipynb`, engine: VGGish via
`torch.hub.load('harritaylor/torchvggish', 'vggish')`):
- Source truth: `<face_output>/log.csv` rows with `status == VALID` (the face pipeline
  log from `vid_to_vec.ipynb`). Only those videos are processed.
- `ffmpeg -vn -ac 1 -ar 16000` video -> wav, cached in `<audio_output>/wav_cache/<split>/`.
- VGGish `model.forward(wav)` -> `(T, 128)` per-window embeddings (~1 per 0.96s).
- Uniform-sample 15 windows (`np.linspace(0, T-1, 15)`). If `T < 15`, take all windows
  then repeat the last window to fill 15 (counted as `pads`). `T == 0` => INVALID `no_audio`.

Output:
- `<audio_output>/<split>/<video_id>.npy` (15, 128) float32
- `<audio_output>/log.csv` (append-only process log / resume source, same semantics as
  `vid_to_vec.ipynb`: skipped iff latest row status == VALID, `force` reprocesses all)

Note: this log is separate from the face pipeline `output/log.csv`.

In [ ]:
import csv
import glob
import os
import subprocess
import time

import numpy as np
import torch

In [ ]:
NUM_SAMPLES = 15
WAV_ARGS = ["-vn", "-ac", "1", "-ar", "16000"]
EMBED_DIM = 128

LOG_COLUMNS = [
    "timestamp",
    "process_time",
    "video_id",
    "split",
    "windows",
    "sampled",
    "pads",
    "status",
    "reason",
]

In [ ]:
def now_ts():
    return time.strftime("%Y-%m-%dT%H:%M:%S")


def make_dirs(path):
    os.makedirs(path, exist_ok=True)


def read_log(log_dir):
    """Return {video_id: last row} from <log_dir>/log.csv."""
    path = os.path.join(log_dir, "log.csv")
    if not os.path.exists(path):
        return {}
    latest = {}
    try:
        with open(path, newline="") as f:
            for row in csv.DictReader(f):
                if row.get("video_id"):
                    latest[row["video_id"]] = row
    except (OSError, csv.Error):
        return latest
    return latest


def append_log(log_dir, row):
    """Append row to <log_dir>/log.csv (independent from the face pipeline log)."""
    path = os.path.join(log_dir, "log.csv")
    new_file = not os.path.exists(path)
    with open(path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_COLUMNS)
        if new_file:
            writer.writeheader()
        writer.writerow(row)

In [ ]:
def get_valid_videos(face_log_root):
    """Return sorted [(split, video_id), ...] with latest row status == VALID.

    Read the face pipeline log produced by vid_to_vec.ipynb.
    """
    log = read_log(face_log_root)
    valid = []
    for video_id, row in log.items():
        if row.get("status") == "VALID" and row.get("split"):
            valid.append((row["split"], video_id))
    return sorted(valid)


def get_video_path(dataset_root, split, video_id):
    """Return path to existing video file (any common container) or None."""
    split_dir = os.path.join(dataset_root, split)
    for ext in (".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v", ".flv", ".wmv", ".mpg", ".mpeg"):
        p = os.path.join(split_dir, f"{video_id}{ext}")
        if os.path.isfile(p):
            return p
    return None

In [ ]:
def convert_to_wav(video_path, wav_path):
    """Convert video -> 16 kHz mono wav with ffmpeg. Return wav_path or None on failure."""
    if os.path.exists(wav_path):
        return wav_path
    make_dirs(os.path.dirname(wav_path))
    command = ["ffmpeg", "-y", "-i", video_path, *WAV_ARGS, wav_path]
    try:
        subprocess.run(command, check=True, capture_output=True, timeout=600)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, OSError) as e:
        print(f"ffmpeg error {os.path.basename(video_path)}: {e}")
        return None
    return wav_path if os.path.exists(wav_path) else None

In [ ]:
def sample_windows(T):
    """Return (indices into T-window feature array or None, pads).

    T >= NUM_SAMPLES: np.linspace(0, T-1, NUM_SAMPLES), pads = 0.
    T <  NUM_SAMPLES: all T windows then repeat last to fill NUM_SAMPLES, pads = NUM_SAMPLES - T.
    T <= 0: (None, 0).
    """
    if T <= 0:
        return None, 0
    if T >= NUM_SAMPLES:
        return np.linspace(0, T - 1, NUM_SAMPLES, dtype=np.int32), 0
    idx = np.minimum(np.arange(NUM_SAMPLES, dtype=np.int32), T - 1)
    return idx, NUM_SAMPLES - T

In [ ]:
def extract_audio_embedding(video_path, wav_path, model):
    """Return (features (15,128) float32 or None, stats dict).

    ffmpeg -> wav, VGGish forward -> (T, 128), uniform-sample to NUM_SAMPLES rows.
    """
    wav = convert_to_wav(video_path, wav_path)
    if wav is None:
        return None, {"reason": "ffmpeg_failed"}

    try:
        feature = model.forward(wav)
    except Exception as e:
        print(f"VGGish error {os.path.basename(video_path)}: {e}")
        return None, {"reason": "vggish_error"}

    feature = np.asarray(feature.cpu().detach().numpy(), dtype=np.float32)
    if feature.ndim != 2:
        return None, {"reason": "bad_feature_shape"}
    T, D = feature.shape
    if D != EMBED_DIM or not np.isfinite(feature).all():
        return None, {"reason": "bad_feature_values"}

    idx, pads = sample_windows(T)
    if idx is None:
        return None, {"frames": T, "reason": "no_audio"}

    arr = feature[idx].astype(np.float32)
    stats = {"frames": T, "sampled": NUM_SAMPLES, "pads": pads}
    return arr, stats

## Run

Process every video whose latest face-log row is VALID. Skips an audio embedding if its
latest `<audio_output>/log.csv` row has status VALID. Set `force=True` to reprocess all.

Paths:
- `root_project` = parent of dataset + output roots (skripsi)
- `face_output_root` = where `vid_to_vec.ipynb` wrote `log.csv` (default `<root_project>/output`)
- `audio_output_root` = this pipeline's output (default `<root_project>/output/audio`)
- `dataset_root` = `<root_project>/first-impressions`

In [ ]:
root_project = "/mnt/4A3ED7573ED73AA1/aa-kuliah/skripsi"
dataset_root = os.path.join(root_project, "first-impressions")
face_output_root = os.path.join(root_project, "output")
audio_output_root = os.path.join(root_project, "output", "audio")
limit = 10
force = False

In [ ]:
print("loading VGGish ...")
model = torch.hub.load("harritaylor/torchvggish", "vggish")
model.eval()
print("model ready")

make_dirs(audio_output_root)
log = read_log(audio_output_root)
valid = get_valid_videos(face_output_root)
print(f"{len(valid)} VALID videos in face log")
if limit:
    valid = valid[:limit]

for split, video_id in valid:
    out_dir = os.path.join(audio_output_root, split)
    make_dirs(out_dir)

    out_path = os.path.join(out_dir, f"{video_id}.npy")
    prev = log.get(video_id)
    if prev and prev.get("status") == "VALID" and not force:
        print(f"  skip {video_id} (VALID)")
        continue

    video_path = get_video_path(dataset_root, split, video_id)
    if video_path is None:
        stats = {"reason": "video_missing"}
        status = "INVALID"
        print(f"  FAIL {video_id} | video_missing")
    else:
        wav_path = os.path.join(audio_output_root, "wav_cache", split, f"{video_id}.wav")
        t0 = time.time()
        arr, stats = extract_audio_embedding(video_path, wav_path, model)
        process_time = round(time.time() - t0, 3)
        if arr is not None:
            np.save(out_path, arr.astype(np.float32))
            status = "VALID"
            print(f"  ok   {video_id} | {process_time}s | {stats}")
        else:
            status = "INVALID"
            reason = stats.get("reason", "unknown")
            print(f"  FAIL {video_id} | {reason} | {process_time}s")

    row = {
        "timestamp": now_ts(),
        "process_time": process_time if "process_time" in dir() else "",
        "video_id": video_id,
        "split": split,
        "windows": stats.get("frames", ""),
        "sampled": stats.get("sampled", ""),
        "pads": stats.get("pads", 0),
        "status": status,
        "reason": stats.get("reason", "") if status != "VALID" else "",
    }
    append_log(audio_output_root, row)
    log[video_id] = row

print("done")

## Integrity check

Validates every `<audio_output>/<split>/*.npy` is shape `(15, 128)`, float32, all finite,
and that every VALID audio-log row has a matching `.npy`.

In [ ]:
def check_npy_integrity(path):
    """Validate one audio .npy: shape (NUM_SAMPLES, EMBED_DIM), float32, all finite.

    Returns (ok: bool, reason: str). reason == "" iff ok.
    """
    try:
        arr = np.load(path, allow_pickle=False)
    except Exception as e:
        return False, f"unreadable: {e}"
    if arr.shape != (NUM_SAMPLES, EMBED_DIM):
        return False, f"bad_shape: {arr.shape}"
    if arr.dtype != np.float32:
        return False, f"bad_dtype: {arr.dtype}"
    if not np.isfinite(arr).all():
        return False, "non_finite_values"
    return True, ""


def check_split_integrity(audio_output_root, split, log=None):
    """Scan <audio_output_root>/<split>/*.npy + cross-check audio log.csv VALID rows.

    Returns (n_ok, problems) where problems is a list of (video_id, reason).
    """
    if log is None:
        log = read_log(audio_output_root)
    split_dir = os.path.join(audio_output_root, split)
    problems = []
    n_ok = 0

    npy_ids = set()
    if os.path.isdir(split_dir):
        for name in sorted(os.listdir(split_dir)):
            if not name.endswith(".npy"):
                continue
            video_id = name[:-4]
            npy_ids.add(video_id)
            ok, reason = check_npy_integrity(os.path.join(split_dir, name))
            if ok:
                n_ok += 1
            else:
                problems.append((video_id, reason))

    for video_id, row in log.items():
        if row.get("split") != split or row.get("status") != "VALID":
            continue
        if video_id not in npy_ids:
            problems.append((video_id, "missing_npy_for_VALID_log"))

    return n_ok, problems

In [ ]:
for split in sorted(set(v[0] for v in get_valid_videos(face_output_root))):
    n_ok, problems = check_split_integrity(audio_output_root, split, log=read_log(audio_output_root))
    print(f"[{split}] ok={n_ok} problems={len(problems)}")
    for video_id, reason in problems:
        print(f"  BAD {video_id}: {reason}")